# 05 — Stability Experiments

Load trained GCN and RGCN, extract embeddings, apply Gaussian noise and dimensional dropout perturbations, measure prediction instability, ranking instability, and compute the Stability Score.

In [ ]:
import sys
sys.path.insert(0, '..')

import os, json
import torch
import numpy as np
import matplotlib.pyplot as plt

from src.preprocessing import load_processed
from src.graph_builder import build_hetero_data, build_homo_data
from src.models import (
    GCNLinkPredictor, RGCNLinkPredictor,
    RELATION_MAP, NODE_TYPES_ORDER, flatten_hetero_graph
)
from src.perturbation import (
    gaussian_noise_perturbation, dimensional_dropout_perturbation,
    run_perturbation_trials
)
from src.stability import full_stability_eval, aggregate_trial_metrics
from src.plotting import (
    plot_stability_curves, plot_topk_overlap,
    plot_delta_p_curves, plot_spearman_curves
)
from src.utils import (
    set_seed, get_device, load_checkpoint,
    DATA_SPLITS, RESULTS_DIR
)
from torch_geometric.data import HeteroData

set_seed(42)
%matplotlib inline

In [ ]:
SEED = 42
device = torch.device('cpu')

id_maps, edges, stats = load_processed()
num_drugs = len(id_maps['drug'])

# Load GCN
homo_ei, _ = build_homo_data(id_maps, edges)
train_edges_ud = torch.load(os.path.join(DATA_SPLITS, 'train_edges_ud.pt'), weights_only=True)

gcn = GCNLinkPredictor(num_drugs, embed_dim=64, dropout=0.2)
gcn = load_checkpoint(gcn, f'gcn_base_s{SEED}', device)
gcn.eval()

# Load RGCN
hetero_data = build_hetero_data(id_maps, edges)
train_hetero = HeteroData()
for ntype in ['drug', 'gene', 'disease']:
    train_hetero[ntype].num_nodes = hetero_data[ntype].num_nodes
train_hetero['drug', 'interacts', 'drug'].edge_index = train_edges_ud.long()
for key in hetero_data.edge_types:
    if key != ('drug', 'interacts', 'drug'):
        train_hetero[key].edge_index = hetero_data[key].edge_index
train_flat_ei, train_flat_et, _ = flatten_hetero_graph(train_hetero, NODE_TYPES_ORDER, RELATION_MAP)

num_nodes_dict = {ntype: len(id_maps[ntype]) for ntype in NODE_TYPES_ORDER}
rgcn = RGCNLinkPredictor(num_nodes_dict, embed_dim=64, num_relations=len(RELATION_MAP), num_bases=2, dropout=0.2)
rgcn = load_checkpoint(rgcn, f'rgcn_base_s{SEED}', device)
rgcn.eval()

print('Models loaded.')

## 1. Extract Learned Drug Embeddings

In [ ]:
with torch.no_grad():
    gcn_z = gcn.encode(train_edges_ud)
    rgcn_z_full = rgcn.encode(train_flat_ei, train_flat_et)
    rgcn_z = rgcn.get_drug_embeddings(rgcn_z_full)

print(f'GCN drug embeddings:  {gcn_z.shape}, mean norm={gcn_z.norm(dim=1).mean():.4f}')
print(f'RGCN drug embeddings: {rgcn_z.shape}, mean norm={rgcn_z.norm(dim=1).mean():.4f}')

## 2. Gaussian Noise Perturbation Experiments

In [ ]:
NOISE_LEVELS = [0.01, 0.05, 0.10, 0.15, 0.20, 0.30]
NUM_TRIALS = 10

gauss_results = {}

for model_name, z in [('gcn', gcn_z), ('rgcn', rgcn_z)]:
    print(f'\n=== {model_name.upper()} Gaussian Noise ===')
    model_results = {}
    trials = run_perturbation_trials(z, gaussian_noise_perturbation, NOISE_LEVELS, NUM_TRIALS)
    
    for sigma in NOISE_LEVELS:
        trial_metrics = []
        for z_pert in trials[sigma]:
            m = full_stability_eval(z, z_pert)
            trial_metrics.append(m)
        agg = aggregate_trial_metrics(trial_metrics)
        model_results[sigma] = agg
        print(f'  σ={sigma:.2f}: StabScore={agg["stability_score_mean"]:.4f}±{agg["stability_score_std"]:.4f}, '
              f'Δp={agg["mean_delta_p_mean"]:.4f}, J@20={agg["mean_jaccard_top20_mean"]:.4f}')
    
    gauss_results[model_name] = model_results

print('\nDone.')

## 3. Dimensional Dropout Perturbation Experiments

In [ ]:
DROPOUT_RATES = [0.05, 0.10, 0.20, 0.30, 0.50]

dropout_results = {}

for model_name, z in [('gcn', gcn_z), ('rgcn', rgcn_z)]:
    print(f'\n=== {model_name.upper()} Dimensional Dropout ===')
    model_results = {}
    trials = run_perturbation_trials(z, dimensional_dropout_perturbation, DROPOUT_RATES, NUM_TRIALS)
    
    for rate in DROPOUT_RATES:
        trial_metrics = []
        for z_pert in trials[rate]:
            m = full_stability_eval(z, z_pert)
            trial_metrics.append(m)
        agg = aggregate_trial_metrics(trial_metrics)
        model_results[rate] = agg
        print(f'  rate={rate:.2f}: StabScore={agg["stability_score_mean"]:.4f}±{agg["stability_score_std"]:.4f}, '
              f'Δp={agg["mean_delta_p_mean"]:.4f}, J@20={agg["mean_jaccard_top20_mean"]:.4f}')
    
    dropout_results[model_name] = model_results

print('\nDone.')

## 4. Plot: Stability Score Curves (Gaussian Noise)

In [ ]:
fig = plot_stability_curves(gauss_results)
plt.title('Stability Score vs Gaussian Noise (σ relative)')
plt.show()

## 5. Plot: Mean |Δp| Curves

In [ ]:
fig = plot_delta_p_curves(gauss_results)
plt.show()

## 6. Plot: Top-K Jaccard Overlap

In [ ]:
fig = plot_topk_overlap(gauss_results, ks=(10, 20, 50))
plt.show()

## 7. Plot: Spearman Rank Correlation

In [ ]:
fig = plot_spearman_curves(gauss_results)
plt.show()

## 8. Plot: Stability Score Curves (Dimensional Dropout)

In [ ]:
fig = plot_stability_curves(dropout_results)
plt.title('Stability Score vs Dimensional Dropout Rate')
plt.show()

## 9. Save All Stability Results

In [ ]:
def convert_keys(d):
    """Convert float keys to strings for JSON serialization."""
    return {str(k): v for k, v in d.items()}

stability_data = {
    'gaussian': {m: convert_keys(r) for m, r in gauss_results.items()},
    'dropout': {m: convert_keys(r) for m, r in dropout_results.items()},
}

with open(os.path.join(RESULTS_DIR, 'stability_results.json'), 'w') as f:
    json.dump(stability_data, f, indent=2)

print('Stability results saved.')